In [1]:
import ardea_sizer as az

base = az.load("base.json")

plane = az.evaluate(base)

print(f"Range:          {plane.range / 1000:8.1f} km")
print(f"Power required: {plane.power_req / 1000:8.2f} kW")
print(f"Power available:{plane.power_available:8.2f} kW")
print(f"Wing area:      {plane.wing_area:8.3f} m²")
print(f"Span:           {plane.span:8.2f} m")
print(f"Chord:          {plane.chord:8.2f} m")

Range:            1049.9 km
Power required:     4.49 kW
Power available:    4.97 kW
Wing area:         0.434 m²
Span:               1.47 m
Chord:              0.29 m


In [3]:
import time
from pathlib import Path

import pandas as pd
import openvsp as vsp

VSP3_FILE = "models/20261012_Convent_BM_V3.vsp3"   # your model
GEOM_SET = 1                  # set from your file: 1 = "Shown"
EXCLUDE = ["CG_Marker"]       # not part of the aircraft, removed from the set (file stays unchanged)

AN = "ParasiteDrag"

# --- load model ---------------------------------------------------------------
vsp.ClearVSPModel()
vsp.ReadVSPFile(str(Path(VSP3_FILE).resolve()))
for name in EXCLUDE:
    for gid in vsp.FindGeomsWithName(name):
        vsp.SetSetFlag(gid, GEOM_SET, False)
vsp.Update()

# --- inputs: copy the settings saved in the file -------------------------------
vsp.SetAnalysisInputDefaults(AN)
settings = vsp.FindContainer("ParasiteDragSettings", 0)
for name in vsp.GetAnalysisInputNames(AN):
    pid = vsp.FindParm(settings, name, "ParasiteDrag")
    if not pid:
        continue
    kind = vsp.GetAnalysisInputType(AN, name)
    if kind == vsp.INT_DATA:
        vsp.SetIntAnalysisInput(AN, name, [int(vsp.GetParmVal(pid))])
    elif kind == vsp.DOUBLE_DATA:
        vsp.SetDoubleAnalysisInput(AN, name, [vsp.GetParmVal(pid)])
for set_input in ("GeomSet", "Set"):
    if set_input in vsp.GetAnalysisInputNames(AN):
        vsp.SetIntAnalysisInput(AN, set_input, [GEOM_SET])
wing_id = vsp.FindGeomsWithName("Main Wing")[0]
s_ref = vsp.GetParmVal(wing_id, "TotalArea", "WingGeom")   # current wing area [m^2]

vsp.SetIntAnalysisInput(AN, "RefFlag", [0])
vsp.SetDoubleAnalysisInput(AN, "Sref", [s_ref])
vsp.PrintAnalysisInputs(AN)

# --- run and time ---------------------------------------------------------------
t0 = time.perf_counter()
rid = vsp.ExecAnalysis(AN)
print(f"\nParasite drag took {time.perf_counter() - t0:.2f} s")

# --- results --------------------------------------------------------------------
getters = {
    vsp.DOUBLE_DATA: vsp.GetDoubleResults,
    vsp.INT_DATA: vsp.GetIntResults,
    vsp.STRING_DATA: vsp.GetStringResults,
}
columns, scalars = {}, {}
for name in vsp.GetAllDataNames(rid):
    getter = getters.get(vsp.GetResultsType(rid, name))
    if getter is None:
        continue
    values = list(getter(rid, name, 0))
    if len(values) == 1:
        scalars[name] = values[0]
    elif values:
        columns[name] = values

n = max(len(v) for v in columns.values())
components = pd.DataFrame({k: v for k, v in columns.items() if len(v) == n})

display(components)                                  # one row per component
display(pd.Series({k: v for k, v in scalars.items() if "CD" in k or "Swet" in k}))


Parasite drag took 1.11 s


,Comp_CD,Comp_Cf,Comp_FFEqn,Comp_FFEqnName,Comp_FFIn,Comp_FFOut,Comp_FineRat,Comp_ID,Comp_Label,Comp_Lref,Comp_PercLam,Comp_PercTotalCD,Comp_Q,Comp_Re,Comp_Roughness,Comp_SurfNum,Comp_Swet,Comp_TawTwRatio,Comp_TeTwRatio,Comp_f
0,0.006811,0.003058,3,Hoerner Streamlined Body,1.0,1.144000,5.784164,QREVHIAIGC,ARDEA_D3_Stack,1.650698,0.0,0.372595,1.0,8.945220e+06,0.0,0,0.778728,1.0,1.0,0.002724
1,0.000966,0.005220,3,Hoerner,1.0,1.206063,0.100028,RCGRJVDWOE,vertical stabilizer,0.082497,0.0,0.052855,1.0,4.470569e+05,0.0,0,0.061386,1.0,1.0,0.000386
2,0.001878,0.005220,3,Hoerner,1.0,1.206063,0.100028,CPCYNWOCMD,Horizontal stabilizer,0.082497,0.0,0.102719,1.0,4.470569e+05,0.0,0,0.119298,1.0,1.0,0.000751
3,0.008625,0.004277,3,Hoerner,1.0,1.231787,0.111291,XHUZTJZXXX,Main Wing,0.234331,0.0,0.471831,1.0,1.269850e+06,0.0,0,0.654769,1.0,1.0,0.003450


Excres_CD_Total            0.0
Geom_CD_Total         0.018279
Swet_Label         S_wet (m^2)
Total_CD_Total        0.018279
dtype: object